# 03 - Addition of weather data
## 1. Load trip data and summarize each city's average starting location and date range

In [1]:
import pandas as pd

trips = pd.read_csv('../data/trips_all_cities_master.csv',
                    usecols=['city', 'date', 'lat_start', 'lng_start'])
trips['date'] = pd.to_datetime(trips['date'])

centers = trips.groupby('city').agg(
    lat=('lat_start', 'mean'),
    lng=('lng_start', 'mean'),
    start=('date', 'min'),
    end=('date', 'max'),
).round(4)
centers

C:\Users\Наташа Ким\AppData\Local\Temp\ipykernel_15336\2212450078.py:12: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(4)


,lat,lng,start,end
city,,,,
Badalona,41.4494,2.2401,2025-04-01,2025-05-31
Brno,49.1986,16.6079,2024-08-01,2024-10-31
Frankfurt,50.1126,8.6732,2025-01-01,2025-03-31
Frydek-Mistek,49.6776,18.3459,2024-08-01,2024-10-31
Havirov,49.7802,18.4389,2024-08-01,2024-10-31
Leon,42.5997,-5.5721,2025-04-01,2025-05-31
Opava,49.9385,17.9035,2024-08-01,2024-10-31
Ostrava,49.8250,18.2409,2024-08-01,2024-10-31


## 2. Download hourly weather from Open-Meteo

In [2]:
#Weather (temperature, rain, wind) may explain part of the demand. Data: Open-Meteo Historical Weather API (CC BY 4.0, based on ERA5 reanalysis). One point per city, at the centre of its stations.
import requests

def get_weather(lat, lng, start, end):
    r = requests.get(
        'https://archive-api.open-meteo.com/v1/archive',
        params={
            'latitude': lat, 'longitude': lng,
            'start_date': start, 'end_date': end,
            'hourly': 'temperature_2m,precipitation,wind_speed_10m,relative_humidity_2m',
            'timezone': 'auto',
        },
        timeout=60,
    )
    r.raise_for_status()
    w = pd.DataFrame(r.json()['hourly'])
    w['time'] = pd.to_datetime(w['time'])
    w['date'] = w['time'].dt.normalize()
    w['hour'] = w['time'].dt.hour
    return w.drop(columns='time')


In [3]:
row = centers.loc['Brno']
test = get_weather(row['lat'], row['lng'], row['start'].date(), row['end'].date())
print(test.shape)
test.head()

(2208, 6)


,temperature_2m,precipitation,wind_speed_10m,relative_humidity_2m,date,hour
0,22.0,0.0,7.7,50,2024-08-01,0
1,23.6,0.0,1.8,46,2024-08-01,1
2,22.9,0.0,3.6,50,2024-08-01,2
3,21.9,0.0,4.5,54,2024-08-01,3
4,21.3,0.1,4.0,59,2024-08-01,4


In [4]:
import time, os

parts = []
for city, row in centers.iterrows():
    w = get_weather(row['lat'], row['lng'], row['start'].date(), row['end'].date())
    w.insert(0, 'city', city)
    parts.append(w)
    time.sleep(1)

weather = pd.concat(parts, ignore_index=True)
os.makedirs('../data/external', exist_ok=True)
weather.to_csv('../data/external/weather_hourly.csv', index=False)
print(weather.shape)

(16128, 7)


In [5]:
print(weather.isna().sum())
print(weather.groupby('city').size())
print(weather.duplicated(['city', 'date', 'hour']).sum(), 'duplicate keys')
print(weather.describe().round(1).T)

city                    0
temperature_2m          0
precipitation           0
wind_speed_10m          0
relative_humidity_2m    0
date                    0
hour                    0
dtype: int64
city
Badalona         1464
Brno             2208
Frankfurt        2160
Frydek-Mistek    2208
Havirov          2208
Leon             1464
Opava            2208
Ostrava          2208
dtype: int64
0 duplicate keys
                        count                        mean  \
temperature_2m        16128.0                        14.5   
precipitation         16128.0                         0.1   
wind_speed_10m        16128.0                        10.0   
relative_humidity_2m  16128.0                        73.8   
date                    16128  2024-11-16 03:49:17.142857   
hour                  16128.0                        11.5   

                                      min                  25%  \
temperature_2m                       -5.4                  9.5   
precipitation                     

In [ ]:
w = weather.copy() # check if the anomalous decline in demand in Czech cities on 2024-09-14 can be explained by weather 
w['day'] = w['date'].dt.date.astype(str)
daily_rain = (w[w['city'].isin(['Brno', 'Ostrava', 'Opava', 'Havirov', 'Frydek-Mistek'])]
              .groupby(['city', 'day'])['precipitation'].sum().round(1)) # total precipitation (mm) in these hours
print(daily_rain.unstack(0).loc['2024-09-11':'2024-09-17'])

city        Brno  Frydek-Mistek  Havirov  Opava  Ostrava
day                                                     
2024-09-11   0.0            0.0      0.1    0.0      0.0
2024-09-12  19.6           18.6     10.4   21.1     12.3
2024-09-13  56.8           43.2     48.6   62.6     48.5
2024-09-14  66.1          100.5     99.7  102.1    107.9
2024-09-15  23.4           66.7     44.2   48.2     46.6
2024-09-16   9.2            8.1      6.4    5.6      6.5
2024-09-17   0.3            0.2      0.0    0.0      0.0


In [ ]:
rain20 = (weather[(weather['date'] == '2024-09-20') & (weather['hour'].between(9, 13))] # Check whether the zero-trip hours on 2024-09-20 (09:00-13:00) can be explained by weather
          .groupby('city')['precipitation'].sum().round(1)) # total precipitation (mm) in these hours
print(rain20)

city
Brno             0.0
Frydek-Mistek    0.0
Havirov          0.0
Opava            0.0
Ostrava          0.0
Name: precipitation, dtype: float64


## 3. Join weather to hourly demand

In [8]:
demand = pd.read_csv('../data/processed/hourly_demand.csv')
demand['date'] = pd.to_datetime(demand['date'])

weather = pd.read_csv('../data/external/weather_hourly.csv')
weather['date'] = pd.to_datetime(weather['date'])

n_before = len(demand)
dw = demand.merge(weather, on=['city', 'date', 'hour'], how='left')

print('Rows before/after:', n_before, len(dw))
print('Missing weather values:', dw[['temperature_2m', 'precipitation',
                                      'wind_speed_10m', 'relative_humidity_2m']].isna().sum().sum())

Rows before/after: 14784 14784
Missing weather values: 0


In [9]:
dw['is_rain'] = (dw['precipitation'] > 0).astype(int)
print(dw.groupby('city')['is_rain'].mean().round(3))

dw.to_csv('../data/processed/hourly_demand_weather.csv', index=False)

city
Badalona         0.112
Brno             0.128
Frankfurt        0.108
Frydek-Mistek    0.157
Havirov          0.138
Leon             0.167
Opava            0.156
Ostrava          0.142
Name: is_rain, dtype: float64
